In [ ]:
### Pré-requisito (para alunos)

### Criar conta no Google

### Obter chave em:
### https://makersuite.google.com/app/apikey

In [ ]:
# Instalar biblioteca oficial
#%pip install -q google-genai 

import google.genai as genai
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Inserir API Key
API_KEY = "GEMINI_API_KEY"

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explique overfitting em português"
)

print(response.text)

In [ ]:
test_cases = {
    "factual": "Quais foram as principais causas da crise financeira de 2008?",
    "technical": "Explique a diferença entre overfitting e underfitting.",
    "decision": "Como você reduziria custos em uma cadeia de suprimentos sem afetar os níveis de serviço?"
}

def generate_prompts(question):
    return {
        "simple": question,
        "structured": f"""
Você é um assistente especializado. Dê uma resposta precisa, estruturada e fundamentada.

Question:
{question}

Answer with:
- explicação
- pontos-chave
- limitações
"""
    }

In [ ]:
def query_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [ ]:
results = []

for category, question in test_cases.items():
    prompts = generate_prompts(question)
    
    for prompt_type, prompt in prompts.items():
        print(f"\n--- {category.upper()} | {prompt_type.upper()} ---\n")
        
        response = query_llm(prompt)
        
        print("Resposta:\n", response[:500], "...\n")
        
        results.append({
            "category": category,
            "prompt_type": prompt_type,
            "prompt": prompt,
            "response": response
        })

In [ ]:
df = pd.DataFrame(results)
df

In [ ]:
def evaluate_response(response):
    print("\n==============================")
    print(response[:800])
    
    factual = int(input("Factual (1-5): "))
    clarity = int(input("Clarity (1-5): "))
    robustness = int(input("Robustness (1-5): "))
    hallucination = int(input("Hallucination (1-5): "))
    
    return {
        "factual": factual,
        "clarity": clarity,
        "robustness": robustness,
        "hallucination": hallucination
    }

evaluations = []

for r in results:
    print(f"\n>>> {r['category']} | {r['prompt_type']}")
    
    scores = evaluate_response(r["response"])
    
    evaluations.append({
        "category": r["category"],
        "prompt_type": r["prompt_type"],
        "response": r["response"],
        **scores
    })

df_eval = pd.DataFrame(evaluations)

In [ ]:
df_eval = pd.DataFrame(evaluations)
df_eval

In [ ]:
df_eval.groupby(["category", "prompt_type"])["factual"].mean().unstack().plot(kind="bar")
plt.title("Precisão factual por categoria e tipo de prompt")
plt.ylabel("Score")
plt.show()

In [ ]:
print("""
ANÁLISE:

1. O prompt estruturado melhorou os resultados?
2. Em que categoria o modelo teve pior desempenho?
3. Foram observadas alucinações?
4. O modelo é consistente?
""")

In [ ]:
print("""
REFLEXÃO (XAI):

- Conseguimos explicar as respostas do modelo?
- Que tipo de explicação seria necessária?
- Como validar decisões baseadas neste modelo?
""")

In [ ]:
print("""
CONCLUSÃO:

- LLM são sensíveis ao prompt
- podem gerar respostas plausíveis mas incorretas
- necessitam validação e explicabilidade

Isto justifica o estudo de XAI nos módulos seguintes.
""")